# Visualization Test

Verifies that molecular structures and trajectories load and display correctly.

In [ ]:
import pathlib

import megane
from megane.parsers.pdb import load_pdb

FIXTURES = (pathlib.Path(".").resolve().parent / "fixtures")
assert FIXTURES.exists(), f"Fixtures not found: {FIXTURES}"
print(f"megane v{megane.__version__}")
print(f"Fixtures: {FIXTURES}")

## Load a PDB structure

In [ ]:
pipe = megane.Pipeline()
s = pipe.add_node(megane.LoadStructure(str(FIXTURES / "1crn.pdb")))
bonds = pipe.add_node(megane.AddBonds(source="distance"))
vp = pipe.add_node(megane.Viewport())
pipe.add_edge(s.out.particle, bonds.inp.particle)
pipe.add_edge(s.out.particle, vp.inp.particle)
pipe.add_edge(bonds.out.bond, vp.inp.bond)

v = megane.MolecularViewer()
v.set_pipeline(pipe)

structure = load_pdb(str(FIXTURES / "1crn.pdb"))
assert structure.n_atoms == 327, f"Expected 327 atoms, got {structure.n_atoms}"
print(f"Loaded 1crn.pdb: {structure.n_atoms} atoms")
v

## Load a trajectory

In [ ]:
traj_pipe = megane.Pipeline()
s = traj_pipe.add_node(megane.LoadStructure(str(FIXTURES / "caffeine_water.pdb")))
t = traj_pipe.add_node(megane.LoadTrajectory(xtc=str(FIXTURES / "caffeine_water_vibration.xtc")))
bonds = traj_pipe.add_node(megane.AddBonds(source="structure"))
vp = traj_pipe.add_node(megane.Viewport())
traj_pipe.add_edge(s.out.particle, t.inp.particle)
traj_pipe.add_edge(s.out.particle, bonds.inp.particle)
traj_pipe.add_edge(s.out.particle, vp.inp.particle)
traj_pipe.add_edge(t.out.traj, vp.inp.traj)
traj_pipe.add_edge(bonds.out.bond, vp.inp.bond)

v_traj = megane.MolecularViewer()
v_traj.set_pipeline(traj_pipe)

assert v_traj.total_frames > 0, "No frames loaded"
print(f"Trajectory: {v_traj.total_frames} frames")

# Navigate frames
v_traj.frame_index = 5
assert v_traj.frame_index == 5
v_traj.frame_index = 0
v_traj

## Multiple viewers side-by-side

In [ ]:
import ipywidgets as widgets

hbox = widgets.HBox([v, v_traj])
assert len(hbox.children) == 2
print("Multiple viewers created successfully")
hbox